#  Events - Bronze Ingestion


## Imports

In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, LongType, StructField, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "retailrocket_events"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "retailrocket"
source_dataset = "events"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("timestamp", LongType(), True),
    StructField("visitorid", StringType(), True),
    StructField("event", StringType(), True),
    StructField("itemid", StringType(), True),
    StructField("transactionid", StringType(), True)
])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

root
 |-- timestamp: long (nullable = true)
 |-- visitorid: string (nullable = true)
 |-- event: string (nullable = true)
 |-- itemid: string (nullable = true)
 |-- transactionid: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

timestamp,visitorid,event,itemid,transactionid,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1432701181734,133468,view,437804,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432700141070,624436,view,212955,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432700032079,251018,view,277505,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432702843159,877542,view,40593,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events
1432701735130,225705,view,219512,null,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/events/events.csv,2026-08-02T21:33:11.000Z,2026-08-03T03:37:54.567Z,336cf555-c549-4ff7-a0aa-2930f9fa5256,retailrocket,events


In [0]:
spark.table(target_table).count()

2756101